# CAPI Automatic Evaluation — BERTScore and ROUGE-L

This notebook computes the automatic evaluation metrics for the five response-generation configurations (**LLM-only, LLM+RAG, and LLM+RAG+PA at Top-1, Top-3, and Top-5**).

* **BERTScore (F1)** — measures the semantic similarity between each generated response and its corresponding ground-truth answer using contextual embeddings.
* **ROUGE-L (F1)** — measures lexical similarity based on the longest common subsequence (LCS) between the generated response and the reference answer.

**Measurement decisions**:

* The **F1 score** is reported for both BERTScore and ROUGE-L to provide a balanced measure of precision and recall.
* Each chatbot response is compared against a single ground-truth answer.
* Scores are computed independently for every response and then aggregated for each configuration.

Requires: `transformers`, `evaluate`, `rouge-score` (`pip install evaluate rouge-score`).


In [ ]:
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
pip install evaluate

In [ ]:
pip install rouge-score

In [ ]:
pip install bert_score

In [ ]:
import pandas as pd
import csv

from pathlib import Path

# Repo root, resolved relative to this file — works regardless of where you run it from.
ROOT = Path(__file__).resolve().parents[1]   # notebook: Path.cwd().parents[0]
DATA = ROOT / "03_Data"

df = pd.read_csv(DATA / "Groundtruth.csv")

In [ ]:
# Display the first few rows of the DataFrame to verify
print(df.head())

In [ ]:

total_data = df[(df['Query'].notna())]

In [ ]:

ground_nan = total_data[(total_data['Ground truth'].isna())]

In [ ]:

filtered_ground_truth = df[(df['Ground truth'].notna())]

In [ ]:

len(total_data)

In [ ]:

top_1 = filtered_ground_truth["Top1"].to_list()
top_3 = filtered_ground_truth["Top 3"].to_list()
top_5 = filtered_ground_truth["Top5"].to_list()
llm_only = filtered_ground_truth["LLMOnly"].to_list()
llm_rag_only = filtered_ground_truth["RAG"].to_list()
ground_truth = filtered_ground_truth["Ground truth"].to_list()

In [ ]:
top_1[0]

In [ ]:
ground_truth[0]

In [ ]:
import re
import string
# menormalisasikan data biar sama formatnya, lowercase, tanpa punctuation dan dibersihkan empty string
def normalize_text(s):
    s = s.lower()

    # replace punctuation with space, not empty string
    s = "".join(" " if ch in string.punctuation else ch for ch in s)

    # fix whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s

In [ ]:
import evaluate

rouge_metric = evaluate.load("rouge")

def calculate_rouge_l(predictions, ground_truths):
    predictions = [normalize_text(x) for x in predictions]
    ground_truths = [normalize_text(x) for x in ground_truths]

    result = rouge_metric.compute(
        predictions=predictions,
        references=ground_truths
    )

    return result["rougeL"] * 100

In [ ]:
# normalisasi hanya untuk membersihkan empty string
def normalize_for_bertscore(s):
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
import evaluate

bertscore = evaluate.load("bertscore")

def calculate_bertscore(predictions, ground_truths):
    predictions = [normalize_for_bertscore(x) for x in predictions]
    ground_truths = [normalize_for_bertscore(x) for x in ground_truths]

    if len(predictions) == 0:
        return 0.0

    '''result = bertscore.compute(
        predictions=predictions,
        references=ground_truths,
        lang="id"
    )'''

    result = bertscore.compute(
        predictions=predictions,
        references=ground_truths,
        lang="id",
    )

    return sum(result["f1"]) / len(result["f1"]) * 100

In [ ]:
import pandas as pd
import evaluate
import re

# Helper function for normalizing text (moved from xAgXPZuAXGIg)
def normalize_for_bertscore(s):
    if s is None: # Handle None values which might result from pd.isna check
        return ""
    return re.sub(r"\s+", " ", str(s)).strip()

bertscore_metric = evaluate.load("bertscore")

def calculate_single_bertscore_f1(prediction, reference):
    # Ensure prediction and reference are strings and not NaN
    if pd.isna(prediction) or pd.isna(reference):
        return None # Or 0.0, depending on how you want to handle missing values

    prediction = normalize_for_bertscore(prediction)
    reference = normalize_for_bertscore(reference)

    # If either is empty after normalization, BERTScore might fail or not be meaningful
    if not prediction or not reference:
        return None

    # bertscore.compute expects lists of predictions and references
    result = bertscore_metric.compute(
        predictions=[prediction],
        references=[reference],
        lang="id"
    )
    return result["f1"][0] * 100 # Return the single F1 score

calculate the BERTScore for each prediction type against the ground truth and add these as new columns to the `filtered_ground_truth` DataFrame.

In [ ]:
# Calculate BERTScore for each prediction type per row
filtered_ground_truth['Top1_BERTScore'] = filtered_ground_truth.apply(lambda row: calculate_single_bertscore_f1(row['Top1'], row['Ground truth']), axis=1)
filtered_ground_truth['Top3_BERTScore'] = filtered_ground_truth.apply(lambda row: calculate_single_bertscore_f1(row['Top 3'], row['Ground truth']), axis=1)
filtered_ground_truth['Top5_BERTScore'] = filtered_ground_truth.apply(lambda row: calculate_single_bertscore_f1(row['Top5'], row['Ground truth']), axis=1)
filtered_ground_truth['LLMOnly_BERTScore'] = filtered_ground_truth.apply(lambda row: calculate_single_bertscore_f1(row['LLMOnly'], row['Ground truth']), axis=1)
filtered_ground_truth['RAG_BERTScore'] = filtered_ground_truth.apply(lambda row: calculate_single_bertscore_f1(row['RAG'], row['Ground truth']), axis=1)

# Display the DataFrame with new BERTScore columns
display(filtered_ground_truth.head())

save the DataFrame with the new BERTScore columns to a CSV file.

In [ ]:
# Save the updated DataFrame to a new CSV file
output_csv_path = book_dir + "PENELITIAN2025/UAT/GroundtruthROUGE_BERTScore_per_query.csv"
filtered_ground_truth.to_csv(output_csv_path, index=False)
print(f"DataFrame with per-query BERTScore saved to: {output_csv_path}")

In [ ]:
top_1_rouge = calculate_rouge_l(top_1, ground_truth)
top_3_rouge = calculate_rouge_l(top_3, ground_truth)
top_5_rouge = calculate_rouge_l(top_5, ground_truth)
llm_only_rouge = calculate_rouge_l(llm_only, ground_truth)
llm_rag_only_rouge = calculate_rouge_l(llm_rag_only, ground_truth)

In [ ]:
print(f"top 1 rougeL: {top_1_rouge}")
print(f"top 3 rougeL: {top_3_rouge}")
print(f"top 5 rougeL: {top_5_rouge}")
print(f"llm only rougeL: {llm_only_rouge}")
print(f"llm rag only rougeL: {llm_rag_only_rouge}")

In [ ]:
top_1_bertscore = calculate_bertscore(top_1, ground_truth)
top_3_bertscore = calculate_bertscore(top_3, ground_truth)
top_5_bertscore = calculate_bertscore(top_5, ground_truth)
llm_only_bertscore = calculate_bertscore(llm_only, ground_truth)
llm_rag_only_bertscore = calculate_bertscore(llm_rag_only, ground_truth)

In [ ]:
print(f"top 1 bertscore: {top_1_bertscore}")
print(f"top 3 bertscore: {top_3_bertscore}")
print(f"top 5 bertscore: {top_5_bertscore}")
print(f"llm only bertscore: {llm_only_bertscore}")
print(f"llm rag only bertscore: {llm_rag_only_bertscore}")

In [ ]:
import pandas as pd

results = {
    "Model": ["Top 1", "Top 3", "Top 5", "LLM Only", "LLM RAG Only"],
    "ROUGE_L": [top_1_rouge, top_3_rouge, top_5_rouge, llm_only_rouge, llm_rag_only_rouge],
    "BERTScore": [top_1_bertscore, top_3_bertscore, top_5_bertscore, llm_only_bertscore, llm_rag_only_bertscore]
}

df_results = pd.DataFrame(results)
print(df_results)

calculate the ROUGE-L for each prediction type against the ground truth and add these as new columns to the `filtered_ground_truth` DataFrame.

In [ ]:
import pandas as pd
import evaluate
import re
import string

# Helper function for normalizing text (copied from sDORUpMDXA52)
def normalize_text_single(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    # replace punctuation with space, not empty string
    s = "".join(" " if ch in string.punctuation else ch for ch in s)
    # fix whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

rouge_metric_single = evaluate.load("rouge")

def calculate_single_rouge_l(prediction, reference):
    # Ensure prediction and reference are strings and not NaN
    if pd.isna(prediction) or pd.isna(reference):
        return None # Or 0.0, depending on how you want to handle missing values

    prediction = normalize_text_single(prediction)
    reference = normalize_text_single(reference)

    # If either is empty after normalization, ROUGE might fail or not be meaningful
    if not prediction or not reference:
        return None

    # rouge_metric_single.compute expects lists of predictions and references
    result = rouge_metric_single.compute(
        predictions=[prediction],
        references=[reference]
    )
    return result["rougeL"] * 100 # Return the single ROUGE-L score

In [ ]:
# Calculate ROUGE-L for each prediction type per row
filtered_ground_truth['Top1_ROUGEL'] = filtered_ground_truth.apply(lambda row: calculate_single_rouge_l(row['Top1'], row['Ground truth']), axis=1)
filtered_ground_truth['Top3_ROUGEL'] = filtered_ground_truth.apply(lambda row: calculate_single_rouge_l(row['Top 3'], row['Ground truth']), axis=1)
filtered_ground_truth['Top5_ROUGEL'] = filtered_ground_truth.apply(lambda row: calculate_single_rouge_l(row['Top5'], row['Ground truth']), axis=1)
filtered_ground_truth['LLMOnly_ROUGEL'] = filtered_ground_truth.apply(lambda row: calculate_single_rouge_l(row['LLMOnly'], row['Ground truth']), axis=1)
filtered_ground_truth['RAG_ROUGEL'] = filtered_ground_truth.apply(lambda row: calculate_single_rouge_l(row['RAG'], row['Ground truth']), axis=1)

# Display the DataFrame with new ROUGE-L columns
display(filtered_ground_truth.head())

save the DataFrame with the new ROUGE-L columns to a CSV file.

In [ ]:
# Save the updated DataFrame to a new CSV file
output_csv_path_rouge = book_dir + "PENELITIAN2025/UAT/Groundtruth_ROUGEL_per_query.csv"
filtered_ground_truth.to_csv(output_csv_path_rouge, index=False)
print(f"DataFrame with per-query ROUGE-L saved to: {output_csv_path_rouge}")